Author: Hasan

Please run this notebook on Google Colab.

In [1]:
!pip install escnn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.5 MB/s eta 0:00:00
  Created wheel for py3nj: filename=py3nj-0.2.1-cp312-cp312-linux_x86_64.whl size=45562 sha256=4c5c7a06f856fcbf0cd26a98bf5bba8abc7393da50fbe5be68b9e2f893643034
  Stored in directory: /root/.cache/pip/wheels/8a/fb/37/2d95d35a34d09f4e0758e53138466650cb6643d8383f4bf605
Successfully built py3nj
  Attempting uninstall: numpy
    Found existing inst

The following code blocks contain a regular $\mathbb{Z}^2$ CNN (equivariant to translations only), a $p4$ G-CNN (equivariant to translations and $90^\circ$ rotations), and a $p4m$ G-CNN (equivariant to translations, $90^\circ$ rotations, and mirror reflections), respectively. Running the code blocks will train the models, printing the train and validation loss per epoch, as well as the validation accuracy per epoch. Once training is complete, the overall test loss, test accuracy, and test accuracy per class will be displayed. This information will also be saved to a results folder, which will include two files: a metrics summary file (which will contain the final results), and a train history file (which will contain the results per epoch).

In [1]:
#Z2 (regular CNN)
import json
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
#learning rate needed to be moved from 0.001 to 0.01 to get better performance, and the number of epochs needed to be increased from 20 to 30 to allow for convergence. With these adjustments, the model achieved a test accuracy of around 80% on CIFAR-10, which is a reasonable result for a simple CNN architecture without data augmentation or advanced regularization techniques.
#batch normalization was also added


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64

NORMALIZE_TRANSFORM = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))


def build_cifar10_transform(train=False, rotation_degrees=0):
    transform_steps = []
    if rotation_degrees > 0:
        if train:
            transform_steps.append(transforms.RandomRotation(rotation_degrees))
        else:
            transform_steps.append(transforms.RandomRotation((rotation_degrees, rotation_degrees)))
    transform_steps.extend([
        transforms.ToTensor(),
        NORMALIZE_TRANSFORM,
    ])
    return transforms.Compose(transform_steps)


DEFAULT_TRANSFORM = build_cifar10_transform()

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_cifar10_datasets(
    root='./data',
    train_transform=DEFAULT_TRANSFORM,
    test_transform=DEFAULT_TRANSFORM,
    download=True,
):
    trainset = torchvision.datasets.CIFAR10(root=root, train=True, download=download, transform=train_transform)
    testset = torchvision.datasets.CIFAR10(root=root, train=False, download=download, transform=test_transform)
    return trainset, testset


def create_dataloader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, generator=None):
    loader_kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': num_workers,
    }
    if generator is not None:
        loader_kwargs['generator'] = generator
    return torch.utils.data.DataLoader(dataset, **loader_kwargs)


def create_train_val_loaders(trainset, batch_size=batch_size, val_fraction=0.1, seed=42):
    val_size = int(len(trainset) * val_fraction)
    if val_size <= 0:
        raise ValueError("val_fraction is too small for the dataset size")
    train_size = len(trainset) - val_size
    g = torch.Generator()
    g.manual_seed(seed)
    train_subset, val_subset = torch.utils.data.random_split(trainset, [train_size, val_size], generator=g)
    trainloader = create_dataloader(train_subset, batch_size=batch_size, shuffle=True, generator=g)
    valloader = create_dataloader(val_subset, batch_size=batch_size, shuffle=False)
    return trainloader, valloader, train_size, val_size


class Net(nn.Module):
    """
    Z2 CNN baseline aligned more closely with the CIFAR-10 setup in
    Cohen and Welling (2016): an All-CNN-style network with 9 convolution
    layers, batch normalization, and global average pooling.
    """
    def __init__(
        self,
        in_channels=3,
        num_classes=10,
        conv_channels=(96, 96, 96, 192, 192, 192, 192, 192, 10),
        kernel_sizes=(3, 3, 3, 3, 3, 3, 3, 1, 1),
        strides=(1, 1, 2, 1, 1, 2, 1, 1, 1),
    ):
        super().__init__()
        if not (len(conv_channels) == len(kernel_sizes) == len(strides) == 9):
            raise ValueError("conv_channels, kernel_sizes, and strides must each contain exactly 9 values")

        self.in_channels = in_channels
        self.num_classes = num_classes
        self.conv_channels = tuple(conv_channels)
        self.kernel_sizes = tuple(kernel_sizes)
        self.strides = tuple(strides)

        layers = []
        current_channels = in_channels
        for out_channels, kernel_size, stride in zip(conv_channels, kernel_sizes, strides):
            padding = kernel_size // 2
            layers.extend([
                nn.Conv2d(current_channels, out_channels, kernel_size, stride=stride, padding=padding),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            ])
            current_channels = out_channels

        self.features = nn.Sequential(*layers)
        self.classifier_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = self.classifier_pool(x)
        x = torch.flatten(x, 1)
        return x


def train(
    net,
    trainloader=None,
    valloader=None,
    epochs=30,
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-4,
    seed=42,
    step_size=10,
    gamma=0.1,
):
    if trainloader is None or valloader is None:
        trainset, testset = get_cifar10_datasets()
        g = torch.Generator()
        g.manual_seed(seed)
        if trainloader is None:
            trainloader, valloader, _, _ = create_train_val_loaders(trainset, batch_size=batch_size, seed=seed)
        elif valloader is None:
            _, valloader, _, _ = create_train_val_loaders(trainset, batch_size=batch_size, seed=seed)

    criterion = nn.CrossEntropyLoss()
    val_criterion = nn.CrossEntropyLoss(reduction='sum')
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    best_val_accuracy = float('-inf')
    best_state = None

    for epoch in range(epochs):
        running_loss = 0.0
        net.train()
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(trainloader)
        history['train_loss'].append(float(avg_loss))

        net.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in valloader:
                images, labels = images.to(device), labels.to(device)
                outputs = net(images)
                val_loss += val_criterion(outputs, labels).item()
                _, predictions = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predictions == labels).sum().item()

        avg_val_loss = val_loss / total
        val_accuracy = 100.0 * correct / total
        history['val_loss'].append(float(avg_val_loss))
        history['val_accuracy'].append(float(val_accuracy))
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(net.state_dict())
        print(
            f'Epoch [{epoch + 1}/{epochs}] Loss: {avg_loss:.3f} '
            f'Val Loss: {avg_val_loss:.4f} Val Acc: {val_accuracy:.2f}%'
        )
        scheduler.step()

    if best_state is not None:
        net.load_state_dict(best_state)

    return history


def evaluate(net, testloader=None, class_names=classes):
    if testloader is None:
        _, testset = get_cifar10_datasets()
        testloader = create_dataloader(testset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(reduction='sum')
    correct_pred = {classname: 0 for classname in class_names}
    total_pred = {classname: 0 for classname in class_names}
    total_loss = 0.0
    total_samples = 0

    net.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            total_loss += criterion(outputs, labels).item()
            total_samples += labels.size(0)
            _, predictions = torch.max(outputs, 1)
            for label, prediction in zip(labels, predictions):
                if label == prediction:
                    correct_pred[class_names[label]] += 1
                total_pred[class_names[label]] += 1

    overall = 100 * sum(correct_pred.values()) / total_samples
    avg_loss = total_loss / total_samples

    print(f'\nTest loss: {avg_loss:.4f}')
    print(f'Overall accuracy: {overall:.1f}%')
    for classname in class_names:
        accuracy = 100 * float(correct_pred[classname]) / total_pred[classname]
        print(f'  {classname:5s}: {accuracy:.1f}%')

    per_class = {
        classname: round(100 * float(correct_pred[classname]) / total_pred[classname], 2)
        for classname in class_names
    }
    return float(avg_loss), float(overall), per_class


def save_results(
    history,
    test_loss,
    test_acc,
    per_class,
    net=None,
    train_size=45000,
    val_size=5000,
    test_size=10000,
    train_rotation_degrees=0,
    test_rotation_degrees=0,
    out_dir='results/cnngeneral_report',
):
    os.makedirs(out_dir, exist_ok=True)

    model = net if net is not None else Net()
    total_params = sum(p.numel() for p in model.parameters())
    conv_filters = list(model.conv_channels) if hasattr(model, 'conv_channels') else []
    summary = {
        'config': {
            'model': 'CNN (Z2)',
            'total_params': total_params,
            'epochs': len(history['train_loss']),
            'batch_size': batch_size,
            'optimizer': 'SGD',
            'lr': 0.01,
            'momentum': 0.9,
            'weight_decay': 5e-4,
            'lr_step_size': 10,
            'lr_gamma': 0.1,
            'train_rotation_degrees': train_rotation_degrees,
            'test_rotation_degrees': test_rotation_degrees,
            'num_conv_layers': len(conv_filters),
            'filters_per_conv_layer': conv_filters,
        },
        'split': {
            'train': train_size,
            'val': val_size,
            'test': test_size,
        },
        'test_metrics': {
            'test_loss': test_loss,
            'test_accuracy': test_acc,
            'per_class_accuracy': per_class,
        },
    }

    with open(os.path.join(out_dir, 'metrics_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    with open(os.path.join(out_dir, 'train_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    print(f'\nSaved report artifacts to: {out_dir}')


if __name__ == '__main__':
    train_rotation_degrees = 0
    test_rotation_degrees = 0

    set_seed(42)
    train_transform = build_cifar10_transform(train=True, rotation_degrees=train_rotation_degrees)
    test_transform = build_cifar10_transform(train=False, rotation_degrees=test_rotation_degrees)
    trainset, testset = get_cifar10_datasets(train_transform=train_transform, test_transform=test_transform)
    trainloader, valloader, train_size, val_size = create_train_val_loaders(trainset, batch_size=batch_size, seed=42)
    testloader = create_dataloader(testset, batch_size=batch_size, shuffle=False)
    net = Net().to(device)
    print(f'Total parameters: {sum(p.numel() for p in net.parameters()):,}')
    history = train(net, trainloader=trainloader, valloader=valloader, epochs=30)
    test_loss, test_acc, per_class = evaluate(net, testloader=testloader)
    save_results(
        history,
        test_loss,
        test_acc,
        per_class,
        net=net,
        train_size=train_size,
        val_size=val_size,
        test_size=len(testset),
        train_rotation_degrees=train_rotation_degrees,
        test_rotation_degrees=test_rotation_degrees,
    )
    torch.save(net.state_dict(), 'cnngeneral.pth')

100%|██████████| 170M/170M [00:05<00:00, 31.0MB/s]


Total parameters: 1,369,738
Epoch [1/30] Loss: 2.303 Test Acc: 10.00%
Epoch [2/30] Loss: 2.303 Test Acc: 10.00%
Epoch [3/30] Loss: 2.303 Test Acc: 10.00%
Epoch [4/30] Loss: 2.303 Test Acc: 10.03%
Epoch [5/30] Loss: 2.303 Test Acc: 10.00%
Epoch [6/30] Loss: 2.303 Test Acc: 10.00%
Epoch [7/30] Loss: 2.303 Test Acc: 10.00%
Epoch [8/30] Loss: 2.303 Test Acc: 9.89%
Epoch [9/30] Loss: 2.303 Test Acc: 9.95%
Epoch [10/30] Loss: 2.303 Test Acc: 9.98%
Epoch [11/30] Loss: 2.303 Test Acc: 9.98%
Epoch [12/30] Loss: 2.303 Test Acc: 9.99%
Epoch [13/30] Loss: 2.303 Test Acc: 9.99%
Epoch [14/30] Loss: 2.303 Test Acc: 9.99%
Epoch [15/30] Loss: 2.303 Test Acc: 9.99%
Epoch [16/30] Loss: 2.303 Test Acc: 9.99%
Epoch [17/30] Loss: 2.303 Test Acc: 9.99%
Epoch [18/30] Loss: 2.303 Test Acc: 9.99%
Epoch [19/30] Loss: 2.303 Test Acc: 9.99%
Epoch [20/30] Loss: 2.303 Test Acc: 9.99%
Epoch [21/30] Loss: 2.303 Test Acc: 9.99%
Epoch [22/30] Loss: 2.303 Test Acc: 9.99%
Epoch [23/30] Loss: 2.303 Test Acc: 9.99%
Epoch [2

In [3]:
#p4 G-CNN
import json
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

from escnn import gspaces
from escnn import nn as enn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64

DEFAULT_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_cifar10_datasets(root='./data', transform=DEFAULT_TRANSFORM, download=True):
    trainset = torchvision.datasets.CIFAR10(root=root, train=True, download=download, transform=transform)
    testset = torchvision.datasets.CIFAR10(root=root, train=False, download=download, transform=transform)
    return trainset, testset


def create_dataloader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, generator=None):
    loader_kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': num_workers,
    }
    if generator is not None:
        loader_kwargs['generator'] = generator
    return torch.utils.data.DataLoader(dataset, **loader_kwargs)


def create_train_val_loaders(trainset, batch_size=batch_size, val_fraction=0.1, seed=42):
    val_size = int(len(trainset) * val_fraction)
    if val_size <= 0:
        raise ValueError("val_fraction is too small for the dataset size")
    train_size = len(trainset) - val_size
    g = torch.Generator()
    g.manual_seed(seed)
    train_subset, val_subset = torch.utils.data.random_split(trainset, [train_size, val_size], generator=g)
    trainloader = create_dataloader(train_subset, batch_size=batch_size, shuffle=True, generator=g)
    valloader = create_dataloader(val_subset, batch_size=batch_size, shuffle=False)
    return trainloader, valloader, train_size, val_size


class GCNN(nn.Module):
    """
    p4-equivariant All-CNN-style network aligned with the CIFAR-10 setup in
    Cohen and Welling (2016). The p4 filter counts are scaled by about sqrt(4)=2
    relative to the planar baseline to keep the parameter count similar.
    """
    def __init__(
        self,
        in_channels=3,
        num_classes=10,
        group_order=4,
        regular_channels=(48, 48, 48, 96, 96, 96, 96, 96, 10),
        kernel_sizes=(3, 3, 3, 3, 3, 3, 3, 1, 1),
        strides=(1, 1, 2, 1, 1, 2, 1, 1, 1),
        input_size=(32, 32),
    ):

        super().__init__()
        if not (len(regular_channels) == len(kernel_sizes) == len(strides) == 9):
            raise ValueError("regular_channels, kernel_sizes, and strides must each contain exactly 9 values")

        self.in_channels = in_channels
        self.num_classes = num_classes
        self.group_order = group_order
        self.regular_channels = tuple(regular_channels)
        self.kernel_sizes = tuple(kernel_sizes)
        self.strides = tuple(strides)
        self.input_size = tuple(input_size)

        self.gspace = gspaces.rot2dOnR2(N=group_order)

        self.in_type = enn.FieldType(self.gspace, in_channels * [self.gspace.trivial_repr])
        blocks = []
        current_type = self.in_type
        for out_channels, kernel_size, stride in zip(regular_channels, kernel_sizes, strides):
            next_type = enn.FieldType(self.gspace, out_channels * [self.gspace.regular_repr])
            blocks.append(
                enn.SequentialModule(
                    enn.R2Conv(
                        current_type,
                        next_type,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=kernel_size // 2,
                        bias=False,
                    ),
                    enn.ReLU(next_type),
                )
            )
            current_type = next_type

        self.blocks = nn.ModuleList(blocks)
        self.gpool = enn.GroupPooling(current_type)
        self.classifier_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = enn.GeometricTensor(x, self.in_type)
        for block in self.blocks:
            x = block(x)
        x = self.gpool(x)
        x = self.classifier_pool(x.tensor)
        x = torch.flatten(x, 1)
        return x


def train(
    net,
    trainloader=None,
    valloader=None,
    epochs=30,
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-4,
    seed=42,
    step_size=10,
    gamma=0.1,
):
    if trainloader is None or valloader is None:
        trainset, _ = get_cifar10_datasets()
        trainloader, valloader, _, _ = create_train_val_loaders(trainset, batch_size=batch_size, seed=seed)

    criterion = nn.CrossEntropyLoss()
    val_criterion = nn.CrossEntropyLoss(reduction='sum')
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    best_val_accuracy = float('-inf')
    best_state = None

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(trainloader)
        history['train_loss'].append(float(avg_train_loss))

        net.eval()
        val_loss = 0.0
        val_samples = 0
        correct = 0
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = net(inputs)
                val_loss += val_criterion(outputs, labels).item()
                val_samples += labels.size(0)
                _, predictions = torch.max(outputs, 1)
                correct += (predictions == labels).sum().item()

        avg_val_loss = val_loss / val_samples
        history['val_loss'].append(float(avg_val_loss))
        val_accuracy = 100.0 * correct / val_samples
        history['val_accuracy'].append(float(val_accuracy))
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(net.state_dict())

        print(
            f'Epoch [{epoch + 1}/{epochs}]  Train Loss: {avg_train_loss:.3f}  '
            f'Val Loss: {avg_val_loss:.4f}  Val Acc: {val_accuracy:.2f}%'
        )
        scheduler.step()

    if best_state is not None:
        net.load_state_dict(best_state)

    return history


def evaluate(net, testloader=None, class_names=classes):
    if testloader is None:
        _, testset = get_cifar10_datasets()
        testloader = create_dataloader(testset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(reduction='sum')

    correct_pred = {classname: 0 for classname in class_names}
    total_pred = {classname: 0 for classname in class_names}
    total_loss = 0.0
    total_samples = 0

    net.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            total_loss += criterion(outputs, labels).item()
            total_samples += labels.size(0)
            _, predictions = torch.max(outputs, 1)
            for label, prediction in zip(labels, predictions):
                if label == prediction:
                    correct_pred[class_names[label]] += 1
                total_pred[class_names[label]] += 1

    overall = 100 * sum(correct_pred.values()) / total_samples
    avg_loss = total_loss / total_samples

    print(f'\nTest loss: {avg_loss:.4f}')
    print(f'Overall accuracy: {overall:.1f}%')
    for classname in class_names:
        accuracy = 100 * float(correct_pred[classname]) / total_pred[classname]
        print(f'  {classname:5s}: {accuracy:.1f}%')

    per_class = {
        classname: round(100 * float(correct_pred[classname]) / total_pred[classname], 2)
        for classname in class_names
    }
    return float(avg_loss), float(overall), per_class


def save_results(
    history,
    test_loss,
    test_acc,
    per_class,
    total_params,
    train_size=45000,
    val_size=5000,
    test_size=10000,
    out_dir='results/gcnngeneral_report',
):
    os.makedirs(out_dir, exist_ok=True)

    conv_filters = list(GCNN().regular_channels)
    summary = {
        'config': {
            'model': 'GCNN (p4)',
            'total_params': total_params,
            'epochs': len(history['train_loss']),
            'batch_size': batch_size,
            'optimizer': 'SGD',
            'lr': 0.01,
            'momentum': 0.9,
            'weight_decay': 5e-4,
            'lr_step_size': 10,
            'lr_gamma': 0.1,
            'num_conv_layers': len(conv_filters),
            'filters_per_conv_layer': conv_filters,
        },
        'split': {
            'train': train_size,
            'val': val_size,
            'test': test_size,
        },
        'test_metrics': {
            'test_loss': test_loss,
            'test_accuracy': test_acc,
            'per_class_accuracy': per_class,
        },
    }

    with open(os.path.join(out_dir, 'metrics_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    with open(os.path.join(out_dir, 'train_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    print(f'\nSaved report artifacts to: {out_dir}')


if __name__ == '__main__':
    set_seed(42)
    trainset, testset = get_cifar10_datasets()
    trainloader, valloader, train_size, val_size = create_train_val_loaders(trainset, batch_size=batch_size, seed=42)
    testloader = create_dataloader(testset, batch_size=batch_size, shuffle=False)
    net = GCNN().to(device)
    total_params = sum(p.numel() for p in net.parameters())
    print(f'Total parameters: {total_params:,}')
    history = train(net, trainloader=trainloader, valloader=valloader, epochs=30)
    test_loss, test_acc, per_class = evaluate(net, testloader=testloader)
    save_results(history, test_loss, test_acc, per_class, total_params, train_size=train_size, val_size=val_size, test_size=len(testset))
    torch.save(net.state_dict(), 'gcnngeneral.pth')



Total parameters: 926,304
Epoch [1/30]  Train Loss: 2.076  Val Loss: 1.9315  Val Acc: 27.38%
Epoch [2/30]  Train Loss: 1.843  Val Loss: 1.7470  Val Acc: 36.96%
Epoch [3/30]  Train Loss: 1.642  Val Loss: 1.4962  Val Acc: 46.22%
Epoch [4/30]  Train Loss: 1.455  Val Loss: 1.3791  Val Acc: 51.64%
Epoch [5/30]  Train Loss: 1.254  Val Loss: 1.1034  Val Acc: 59.38%
Epoch [6/30]  Train Loss: 1.069  Val Loss: 0.9840  Val Acc: 65.52%
Epoch [7/30]  Train Loss: 0.911  Val Loss: 0.8776  Val Acc: 69.28%
Epoch [8/30]  Train Loss: 0.807  Val Loss: 0.8128  Val Acc: 71.72%
Epoch [9/30]  Train Loss: 0.720  Val Loss: 0.7318  Val Acc: 74.62%
Epoch [10/30]  Train Loss: 0.650  Val Loss: 0.7356  Val Acc: 74.72%
Epoch [11/30]  Train Loss: 0.473  Val Loss: 0.6278  Val Acc: 78.68%
Epoch [12/30]  Train Loss: 0.439  Val Loss: 0.6253  Val Acc: 78.64%
Epoch [13/30]  Train Loss: 0.420  Val Loss: 0.6225  Val Acc: 78.98%
Epoch [14/30]  Train Loss: 0.406  Val Loss: 0.6162  Val Acc: 79.02%
Epoch [15/30]  Train Loss: 0.39

In [4]:
#p4m G-CNN
import json
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

from escnn import gspaces
from escnn import nn as enn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64

DEFAULT_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_cifar10_datasets(root='./data', transform=DEFAULT_TRANSFORM, download=True):
    trainset = torchvision.datasets.CIFAR10(root=root, train=True, download=download, transform=transform)
    testset = torchvision.datasets.CIFAR10(root=root, train=False, download=download, transform=transform)
    return trainset, testset


def create_dataloader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, generator=None):
    loader_kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': num_workers,
    }
    if generator is not None:
        loader_kwargs['generator'] = generator
    return torch.utils.data.DataLoader(dataset, **loader_kwargs)


def create_train_val_loaders(trainset, batch_size=batch_size, val_fraction=0.1, seed=42):
    val_size = int(len(trainset) * val_fraction)
    if val_size <= 0:
        raise ValueError("val_fraction is too small for the dataset size")
    train_size = len(trainset) - val_size
    g = torch.Generator()
    g.manual_seed(seed)
    train_subset, val_subset = torch.utils.data.random_split(trainset, [train_size, val_size], generator=g)
    trainloader = create_dataloader(train_subset, batch_size=batch_size, shuffle=True, generator=g)
    valloader = create_dataloader(val_subset, batch_size=batch_size, shuffle=False)
    return trainloader, valloader, train_size, val_size


class GCNNP4M(nn.Module):
    """
    p4m-equivariant All-CNN-style network aligned with the CIFAR-10 setup in
    Cohen and Welling (2016). The p4m filter counts are scaled by about
    sqrt(8) relative to the planar baseline to keep the parameter count similar.
    """

    def __init__(
        self,
        in_channels=3,
        num_classes=10,
        group_order=4,
        regular_channels=(32, 32, 32, 64, 64, 64, 64, 64, 10),
        kernel_sizes=(3, 3, 3, 3, 3, 3, 3, 1, 1),
        strides=(1, 1, 2, 1, 1, 2, 1, 1, 1),
        input_size=(32, 32),
    ):
        super().__init__()
        if not (len(regular_channels) == len(kernel_sizes) == len(strides) == 9):
            raise ValueError("regular_channels, kernel_sizes, and strides must each contain exactly 9 values")

        self.in_channels = in_channels
        self.num_classes = num_classes
        self.group_order = group_order
        self.regular_channels = tuple(regular_channels)
        self.kernel_sizes = tuple(kernel_sizes)
        self.strides = tuple(strides)
        self.input_size = tuple(input_size)

        self.gspace = gspaces.flipRot2dOnR2(N=group_order)

        self.in_type = enn.FieldType(self.gspace, in_channels * [self.gspace.trivial_repr])
        blocks = []
        current_type = self.in_type
        for out_channels, kernel_size, stride in zip(regular_channels, kernel_sizes, strides):
            next_type = enn.FieldType(self.gspace, out_channels * [self.gspace.regular_repr])
            blocks.append(
                enn.SequentialModule(
                    enn.R2Conv(
                        current_type,
                        next_type,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=kernel_size // 2,
                        bias=False,
                    ),
                    enn.ReLU(next_type),
                )
            )
            current_type = next_type

        self.blocks = nn.ModuleList(blocks)
        self.gpool = enn.GroupPooling(current_type)
        self.classifier_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = enn.GeometricTensor(x, self.in_type)
        for block in self.blocks:
            x = block(x)
        x = self.gpool(x)
        x = self.classifier_pool(x.tensor)
        x = torch.flatten(x, 1)
        return x


def train(
    net,
    trainloader=None,
    valloader=None,
    epochs=30,
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-4,
    seed=42,
    step_size=10,
    gamma=0.1,
):
    if trainloader is None or valloader is None:
        trainset, _ = get_cifar10_datasets()
        trainloader, valloader, _, _ = create_train_val_loaders(trainset, batch_size=batch_size, seed=seed)

    criterion = nn.CrossEntropyLoss()
    val_criterion = nn.CrossEntropyLoss(reduction='sum')
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    best_val_accuracy = float('-inf')
    best_state = None

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(trainloader)
        history['train_loss'].append(float(avg_train_loss))

        net.eval()
        val_loss = 0.0
        val_samples = 0
        correct = 0
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = net(inputs)
                val_loss += val_criterion(outputs, labels).item()
                val_samples += labels.size(0)
                _, predictions = torch.max(outputs, 1)
                correct += (predictions == labels).sum().item()

        avg_val_loss = val_loss / val_samples
        history['val_loss'].append(float(avg_val_loss))
        val_accuracy = 100.0 * correct / val_samples
        history['val_accuracy'].append(float(val_accuracy))
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(net.state_dict())

        print(
            f'Epoch [{epoch + 1}/{epochs}]  Train Loss: {avg_train_loss:.3f}  '
            f'Val Loss: {avg_val_loss:.4f}  Val Acc: {val_accuracy:.2f}%'
        )
        scheduler.step()

    if best_state is not None:
        net.load_state_dict(best_state)

    return history


def evaluate(net, testloader=None, class_names=classes):
    if testloader is None:
        _, testset = get_cifar10_datasets()
        testloader = create_dataloader(testset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(reduction='sum')

    correct_pred = {classname: 0 for classname in class_names}
    total_pred = {classname: 0 for classname in class_names}
    total_loss = 0.0
    total_samples = 0

    net.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            total_loss += criterion(outputs, labels).item()
            total_samples += labels.size(0)
            _, predictions = torch.max(outputs, 1)
            for label, prediction in zip(labels, predictions):
                if label == prediction:
                    correct_pred[class_names[label]] += 1
                total_pred[class_names[label]] += 1

    overall = 100 * sum(correct_pred.values()) / total_samples
    avg_loss = total_loss / total_samples

    print(f'\nTest Loss: {avg_loss:.4f}')
    print(f'Overall Accuracy: {overall:.1f}%')
    for classname in class_names:
        accuracy = 100 * float(correct_pred[classname]) / total_pred[classname]
        print(f'  {classname:5s}: {accuracy:.1f}%')

    per_class = {
        classname: round(100 * float(correct_pred[classname]) / total_pred[classname], 2)
        for classname in class_names
    }
    return float(avg_loss), float(overall), per_class


def save_results(
    history,
    test_loss,
    test_acc,
    per_class,
    total_params,
    train_size=45000,
    val_size=5000,
    test_size=10000,
    out_dir='results/gcnn_p4mgeneral_report',
):
    os.makedirs(out_dir, exist_ok=True)

    conv_filters = list(GCNNP4M().regular_channels)
    summary = {
        'config': {
            'model': 'GCNN (p4m)',
            'total_params': total_params,
            'epochs': len(history['train_loss']),
            'batch_size': batch_size,
            'optimizer': 'SGD',
            'lr': 0.01,
            'momentum': 0.9,
            'weight_decay': 5e-4,
            'lr_step_size': 10,
            'lr_gamma': 0.1,
            'num_conv_layers': len(conv_filters),
            'filters_per_conv_layer': conv_filters,
        },
        'split': {
            'train': train_size,
            'val': val_size,
            'test': test_size,
        },
        'test_metrics': {
            'test_loss': test_loss,
            'test_accuracy': test_acc,
            'per_class_accuracy': per_class,
        },
    }

    with open(os.path.join(out_dir, 'metrics_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    with open(os.path.join(out_dir, 'train_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    print(f'\nSaved report artifacts to: {out_dir}')


if __name__ == '__main__':
    set_seed(42)
    trainset, testset = get_cifar10_datasets()
    trainloader, valloader, train_size, val_size = create_train_val_loaders(trainset, batch_size=batch_size, seed=42)
    testloader = create_dataloader(testset, batch_size=batch_size, shuffle=False)
    net = GCNNP4M().to(device)
    total_params = sum(p.numel() for p in net.parameters())
    print(f'Total parameters: {total_params:,}')
    history = train(net, trainloader=trainloader, valloader=valloader, epochs=30)
    test_loss, test_acc, per_class = evaluate(net, testloader=testloader)
    save_results(history, test_loss, test_acc, per_class, total_params, train_size=train_size, val_size=val_size, test_size=len(testset))
    torch.save(net.state_dict(), 'gcnn_p4mgeneral.pth')



Total parameters: 824,896
Epoch [1/30]  Train Loss: 2.138  Val Loss: 1.9975  Val Acc: 24.14%
Epoch [2/30]  Train Loss: 1.907  Val Loss: 1.8466  Val Acc: 30.98%
Epoch [3/30]  Train Loss: 1.720  Val Loss: 1.6546  Val Acc: 38.20%
Epoch [4/30]  Train Loss: 1.585  Val Loss: 1.5263  Val Acc: 45.20%
Epoch [5/30]  Train Loss: 1.459  Val Loss: 1.4427  Val Acc: 48.38%
Epoch [6/30]  Train Loss: 1.317  Val Loss: 1.3129  Val Acc: 52.34%
Epoch [7/30]  Train Loss: 1.161  Val Loss: 1.1674  Val Acc: 59.40%
Epoch [8/30]  Train Loss: 1.045  Val Loss: 0.9994  Val Acc: 65.36%
Epoch [9/30]  Train Loss: 0.944  Val Loss: 0.9256  Val Acc: 67.86%
Epoch [10/30]  Train Loss: 0.858  Val Loss: 0.8766  Val Acc: 69.36%
Epoch [11/30]  Train Loss: 0.689  Val Loss: 0.7423  Val Acc: 73.76%
Epoch [12/30]  Train Loss: 0.659  Val Loss: 0.7267  Val Acc: 74.06%
Epoch [13/30]  Train Loss: 0.643  Val Loss: 0.7334  Val Acc: 73.90%
Epoch [14/30]  Train Loss: 0.632  Val Loss: 0.7196  Val Acc: 74.54%
Epoch [15/30]  Train Loss: 0.61